# Neuron and Activation Functions

AI & ML Foundations: Neural networks and backpropagation · *Mathematics for AI and Machine Learning* by Fuheng Wu

One neuron y = phi(w x + b): how weight, bias and the activation (sigmoid, tanh, ReLU, GELU) shape the output and its gradient.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/neuron-activation/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
import torch.nn.functional as F

# The demo's defaults: one neuron y = phi(w x + b) with w = 1.5, b = 0.5
w, b = 1.5, 0.5
x = torch.linspace(-4, 4, 801, dtype=torch.float64, requires_grad=True)

activations = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "relu": F.relu,
    "gelu": F.gelu,
}

print(f"{'activation':10s} {'max |dy/dx|':>12s} {'at x':>7s} {'saturated share':>16s}  output range")
for name, phi in activations.items():
    y = phi(w * x + b)
    (dy,) = torch.autograd.grad(y.sum(), x)            # dy/dx = w phi'(w x + b), by autograd
    peak = dy.abs().max()
    flat = (dy.abs() < 0.05 * peak).double().mean().item()
    print(f"{name:10s} {peak.item():12.4f} {x[dy.abs().argmax()].item():7.2f} {flat:15.0%}   "
          f"[{y.min().item():+.3f}, {y.max().item():+.3f}]")

# ── Check autograd against the closed-form derivatives ──────────────────────
z = w * x.detach() + b
s = torch.sigmoid(z)
formulas = {
    "sigmoid": w * s * (1 - s),
    "tanh": w * (1 - torch.tanh(z) ** 2),
    "relu": w * (z > 0).double(),
    "gelu": w * (0.5 * (1 + torch.erf(z / math.sqrt(2))) + z * torch.exp(-0.5 * z ** 2) / math.sqrt(2 * math.pi)),
}
for name, phi in activations.items():
    (dy,) = torch.autograd.grad(phi(w * x + b).sum(), x)
    print(f"{name:8s} autograd matches the formula: {torch.allclose(dy, formulas[name])}")

# The steepest point of sigmoid/tanh is where w x + b = 0
print(f"\nsigmoid/tanh are steepest at x = -b/w = {-b / w:.4f}; sigmoid's slope there is w/4 = {w / 4}")
# A ReLU unit whose input is always negative gets zero gradient: it is "dead"
print("ReLU with b = -10 on x in [-4, 4]: gradient share that is zero =",
      (torch.autograd.grad(F.relu(w * x - 10).sum(), x)[0] == 0).double().mean().item())

activation  max |dy/dx|    at x  saturated share  output range
sigmoid          0.3750   -0.33             27%   [+0.004, +0.998]
tanh             1.5000   -0.33             64%   [-1.000, +1.000]
relu             1.5000   -0.33             46%   [+0.000, +6.500]
gelu             1.6934    0.61             29%   [-0.170, +6.500]
sigmoid  autograd matches the formula: True
tanh     autograd matches the formula: True
relu     autograd matches the formula: True
gelu     autograd matches the formula: True

sigmoid/tanh are steepest at x = -b/w = -0.3333; sigmoid's slope there is w/4 = 0.375
ReLU with b = -10 on x in [-4, 4]: gradient share that is zero = 1.0
